# NASA APOD 1995: Mining Cosmic Origins

### Sobre o Projeto
Este projeto apresenta uma análise exploratória e técnica das primeiras 100 imagens do programa **Astronomy Picture of the Day (APOD)** da NASA, datadas de 1995. Através de um fluxo de Ciência de Dados, transformamos metadados brutos e arquivos de imagem em um dataset enriquecido com categorização científica e métricas técnicas.



###  Principais Insights Extraídos
* **Domínio Temático**: 60% do acervo inicial foca no **Sistema Solar**, seguido por 23% de Galáxias.
* **Perfil de Mídia**: 100% dos arquivos são GIFs com resolução média de **395x303 pixels**, refletindo os padrões tecnológicos da web em 1995.
* **Segurança Legal**: **92%** das imagens são de Domínio Público, garantindo ampla liberdade para experimentação.

###  Metodologia Aplicada
O projeto seguiu quatro etapas rigorosas:
1. **Ingestão e Auditoria**: Mapeamento de caminhos no ambiente Kaggle e validação de integridade.
2. **Processamento de Imagem**: Extração de metadados técnicos (resolução e animação) usando a biblioteca `Pillow`.
3. **Mineração de Texto (NLP)**: Categorização automática de objetos astronômicos baseada nas explicações científicas.
4. **Análise Estatística**: Correlação entre categorias de objetos e atributos físicos das imagens.

###  Estrutura dos Arquivos
* `sample_images_metadata.csv`: Dados originais da NASA.
* `nasa_apod_analise_final.csv`: Dataset enriquecido gerado por este projeto, pronto para modelos de Machine Learning.

---
**Instituição:** IFRN 

## 1. Preparação do Ambiente e Inspeção de Metadados

Nesta etapa inicial, estabelecemos a conexão com os diretórios de dados do Kaggle e realizamos uma auditoria técnica nos metadados. O objetivo é garantir que os arquivos físicos correspondam aos registros do CSV e entender as limitações legais de uso das imagens.

**O que estamos fazendo:**
1.  **Mapeamento de Caminhos**: Definindo as constantes de diretório do ambiente Kaggle.
2.  **Tratamento de Licenciamento**: Criando uma classificação binária entre imagens de **Domínio Público** (NASA) e imagens com **Direitos Autorais** (Copyright), baseando-se na presença de valores nulos na coluna `copyright`.
3.  **Auditoria de Arquivos**: Verificando via código se cada imagem listada no arquivo `.csv` realmente existe na pasta de imagens.
4.  **Perfil Estatístico**: Gerando um resumo quantitativo e visual sobre a composição do dataset.



---
**Ferramentas utilizadas:** `pandas` para manipulação tabular, `os` para verificação de sistema de arquivos e `seaborn/matplotlib` para visualização.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# 1. Configuração dos caminhos do Kaggle
BASE_PATH = '/kaggle/input/nasa-apod-sample-images-100-astronomy-pictures/'
METADATA_PATH = os.path.join(BASE_PATH, 'sample_images_metadata.csv')
IMAGES_DIR = os.path.join(BASE_PATH, 'nasa_apod_sample_images/')

# 2. Carregamento dos metadados
df = pd.read_csv(METADATA_PATH)

# 3. Classificação de Licenciamento
# Regra: Se 'copyright' for nulo (NaN), a imagem é Domínio Público (NASA)
df['licenca_status'] = df['copyright'].apply(
    lambda x: 'Domínio Público' if pd.isna(x) else 'Copyright (Crédito Necessário) ⚠️'
)

# 4. Verificação de integridade dos arquivos de imagem
df['arquivo_existe'] = df['local_filename'].apply(
    lambda x: os.path.exists(os.path.join(IMAGES_DIR, x))
)

# 5. Exibição das Estatísticas Iniciais
print(f"--- Relatório de Integridade ---")
print(f"Total de registros no CSV: {len(df)}")
print(f"Imagens encontradas no diretório: {df['arquivo_existe'].sum()}")

print("\n--- Distribuição de Licenciamento ---")
status_counts = df['licenca_status'].value_counts()
status_pct = df['licenca_status'].value_counts(normalize=True) * 100
for status, count in status_counts.items():
    print(f"{status}: {count} ({status_pct[status]:.1f}%)")

# 6. Visualização da Composição do Dataset
plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")
sns.countplot(data=df, x='licenca_status', palette=['#4CAF50', '#FF9800'])
plt.title('Composição Legal do Dataset NASA APOD')
plt.xlabel('Status de Direitos Autorais')
plt.ylabel('Quantidade de Imagens')
plt.show()

# Exibição de amostra dos dados
display(df[['date', 'title', 'licenca_status', 'local_filename']].head())

## 2. Processamento Técnico e Engenharia de Atributos Visuais

Nesta fase, avançamos para o tratamento do dado não estruturado (a imagem). O objetivo é realizar a leitura técnica dos arquivos (incluindo os GIFs) para extrair metadados físicos que não constam no CSV original, como resoluções reais e modos de cor, permitindo uma análise mais profunda da tecnologia da época.

**Ações realizadas:**
1.  **Extração de Formato e Resolução**: Identificação via código se o arquivo é um GIF (estático ou animado), JPG ou PNG, e captura das dimensões em pixels.
2.  **Análise de Profundidade de Cor**: Verificação do modo de cor (RGB, Escala de Cinza ou Paletizado), revelando a tecnologia de captura.
3.  **Engenharia de Atributos Temporais**: Conversão da coluna de data para o formato `datetime` para permitir agrupamentos e análise de sazonalidade.
4.  **Detecção de Animação**: Identificação específica de quais GIFs são sequências animadas, dado importante para o engajamento visual do projeto.



[Image of digital image processing flow]

In [ ]:
import pandas as pd
from PIL import Image
import os

# Configurações de caminhos (Kaggle)
BASE_PATH = '/kaggle/input/nasa-apod-sample-images-100-astronomy-pictures/'
IMAGES_DIR = os.path.join(BASE_PATH, 'nasa_apod_sample_images/')
METADATA_PATH = os.path.join(BASE_PATH, 'sample_images_metadata.csv')

# 1. Carregamento e Conversão Temporal
df = pd.read_csv(METADATA_PATH)
df['date'] = pd.to_datetime(df['date'])

def extrair_dados_imagem(row):
    """Função para abrir o arquivo e extrair metadados técnicos."""
    path = os.path.join(IMAGES_DIR, row['local_filename'])
    try:
        with Image.open(path) as img:
            return pd.Series({
                'largura': img.width,
                'altura': img.height,
                'formato_tecnico': img.format,
                'modo_cor': img.mode,
                'is_animado': getattr(img, "is_animated", False)
            })
    except Exception:
        return pd.Series({
            'largura': None, 'altura': None, 
            'formato_tecnico': 'Erro Leitura', 'modo_cor': None, 
            'is_animado': False
        })

# 2. Processamento em lote das imagens
print("Extraindo atributos técnicos das 100 imagens...")
df_extra = df.apply(extrair_dados_imagem, axis=1)
df = pd.concat([df, df_extra], axis=1)

# 3. Engenharia de Atributos Temporais
df['mes'] = df['date'].dt.month
df['dia_semana'] = df['date'].dt.day_name()

# 4. Resumo Técnico do Processamento
print("\n--- Relatório Técnico de Mídia ---")
print(f"Formatos Identificados:\n{df['formato_tecnico'].value_counts()}")
print(f"\nGIFs Animados Detectados: {df['is_animado'].sum()}")
print(f"Resolução Média: {df['largura'].mean():.0f}x{df['altura'].mean():.0f} pixels")

# Salvando checkpoint da Etapa 2
df.to_csv('metadata_processado_etapa2.csv', index=False)
display(df[['date', 'title', 'formato_tecnico', 'is_animado', 'largura', 'altura']].head())

## 3. Análise Exploratória de Dados (EDA) e Categorização Científica

Nesta fase, transformamos as descrições científicas textuais e os atributos técnicos em informações estatísticas. O objetivo é realizar a mineração de termos astronômicos para identificar os temas predominantes no início do projeto APOD e correlacionar a natureza do objeto com os dados técnicos coletados.

**Ações realizadas:**
1.  **Mineração de Texto e Categorização**: Implementação de um classificador baseado em palavras-chave para agrupar as imagens em categorias como Galáxias, Nebulosas, Sistema Solar e Missões Espaciais.
2.  **Análise de Sazonalidade**: Estudo da distribuição temporal das postagens para verificar o volume de dados disponível por mês em 1995.
3.  **Análise de Resolução por Categoria**: Cruzamento de dados para entender se determinados objetos celestes (como galáxias) apresentavam resoluções maiores ou menores na época.
4.  **Resumo Estatístico Final**: Consolidação das métricas para suportar decisões em modelos futuros de Computer Vision ou Data Science.

[Image of exploratory data analysis charts]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carregamento do checkpoint da etapa anterior
# Certifique-se de que o arquivo 'metadata_processado_etapa2.csv' existe no diretório
try:
    df = pd.read_csv('metadata_processado_etapa2.csv')
except FileNotFoundError:
    # Fallback caso o arquivo tenha nome diferente ou precise ser carregado do original
    df = pd.read_csv('/kaggle/input/nasa-apod-sample-images-100-astronomy-pictures/sample_images_metadata.csv')
    df['date'] = pd.to_datetime(df['date'])

# 2. Lógica de Categorização Científica (Keyword Mining)
def categorizar_objeto(texto):
    texto = str(texto).lower()
    if 'galaxy' in texto: return 'Galáxia'
    elif 'nebula' in texto: return 'Nebulosa'
    elif any(word in texto for word in ['planet', 'earth', 'moon', 'mars', 'jupiter', 'sun', 'comet']): return 'Sistema Solar'
    elif any(word in texto for word in ['star', 'cluster', 'supernova', 'pulsar']): return 'Estrelas/Aglomerados'
    elif any(word in texto for word in ['apollo', 'hubble', 'spacecraft', 'shuttle']): return 'Missões/Tecnologia'
    else: return 'Outros Fenômenos'

df['categoria'] = df['explanation'].apply(categorizar_objeto)

# 3. Visualização: Distribuição de Categorias
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
cat_order = df['categoria'].value_counts().index
sns.countplot(data=df, y='categoria', order=cat_order, palette='magma')
plt.title('Distribuição de Objetos Astronômicos (NASA APOD 1995)', fontsize=14)
plt.xlabel('Quantidade de Imagens')
plt.ylabel('Categoria')
plt.savefig('distribuicao_categorias.png')
plt.show()

# 4. Análise Temporal (Volume por Mês)
df['mes_nome'] = pd.to_datetime(df['date']).dt.month_name()
mes_order = ['June', 'July', 'August', 'September']
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='mes_nome', order=mes_order, palette='viridis')
plt.title('Volume de Publicações por Mês em 1995')
plt.ylabel('Total')
plt.xlabel('Mês')
plt.savefig('volume_mensal.png')
plt.show()

# 5. Tabela de Resumo Estatístico
print("--- Métricas por Categoria ---")
resumo = df.groupby('categoria').agg({
    'largura': ['mean', 'count'],
    'altura': 'mean'
}).round(1)
resumo.columns = ['Largura Média', 'Total Imagens', 'Altura Média']
print(resumo.sort_values(by='Total Imagens', ascending=False))

# Exportação do dataset final enriquecido
df.to_csv('nasa_apod_analise_final.csv', index=False)
print("\nDataset 'nasa_apod_analise_final.csv' gerado com sucesso.")

## 4. Conclusão e Insights Estratégicos

Nesta etapa final, consolidamos todo o conhecimento extraído das 100 imagens da NASA. O fluxo seguiu o rigor metodológico: desde a limpeza de metadados, passando pelo processamento técnico dos GIFs, até a categorização científica via mineração de texto.

**Principais Descobertas:**
* **Predomínio Temático**: O Sistema Solar é o tema central (60%), indicando um forte apelo educacional inicial.
* **Perfil Tecnológico**: Todas as imagens da amostra são GIFs estáticos com resolução média de 395x303 pixels, refletindo as limitações de largura de banda de 1995.
* **Consistência Legal**: 92% do dataset é de Domínio Público, o que viabiliza o uso deste projeto para fins acadêmicos e comerciais sem restrições severas.
* **Relação de Aspecto**: Objetos como "Missões/Tecnologia" tendem a ter uma altura média maior (350.4px) comparado a fenômenos como "Galáxias".

---
**Status do Projeto:** Finalizado e Pronto para Produção.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Carregamento dos resultados finais
df_final = pd.read_csv('nasa_apod_analise_final.csv')

# 2. Visualização de Comparação de Resolução (Boxplot)
plt.figure(figsize=(12, 7))
sns.boxplot(data=df_final, x='categoria', y='largura', palette='Set2')
plt.title('Variação de Resolução (Largura) por Categoria de Objeto', fontsize=14)
plt.xticks(rotation=45)
plt.ylabel('Largura em Pixels')
plt.xlabel('Categoria Científica')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# 3. Gráfico de Pizza: Composição do Catálogo (Share of Voice)
plt.figure(figsize=(8, 8))
composicao = df_final['categoria'].value_counts()
plt.pie(composicao, labels=composicao.index, autopct='%1.1f%%', 
        startangle=140, colors=sns.color_palette('viridis', len(composicao)))
plt.title('Composição do Acervo NASA APOD (1995)', fontsize=14)
plt.show()

# 4. Mensagem de Encerramento
print("--- PROJETO CONCLUÍDO ---")
print(f"Total de registros processados: {len(df_final)}")
print(f"Arquivo final disponível: 'nasa_apod_analise_final.csv'")
print("As visualizações acima resumem o perfil técnico e científico da amostra.")

## 5. Conclusão e Entrega de Insights

O projeto **NASA APOD Sample Analysis** foi concluído com sucesso, cobrindo desde a ingestão de dados brutos até a mineração de texto para categorização astronômica. 

**Resultados Consolidados:**
1.  **Perfil Técnico**: Identificamos que 100% da amostra de 1995 utiliza o formato GIF, com uma resolução média de 395x303 pixels, refletindo a infraestrutura da web daquela época.
2.  **Foco Temático**: O Sistema Solar representa 60% do conteúdo, seguido por Galáxias (23%), o que sugere uma priorização de objetos mais conhecidos/próximos no início do projeto.
3.  **Segurança Jurídica**: Confirmamos que 92% das imagens são de Domínio Público, facilitando o reaproveitamento dos dados.
4.  **Produto Final**: O arquivo `nasa_apod_analise_final.csv` agora contém não apenas os metadados originais, mas também a classificação dos objetos e as dimensões físicas das imagens.

---
**Próximos Passos Sugeridos:**
* Implementação de um modelo de **Deep Learning** (CNN) para classificação automática de categorias.
* Análise de **Sentimento/Entropia** das explicações científicas para medir a complexidade do texto.